# Project: Personal Chef agent

**Module:** Module 1 — Agent Fundamentals  
**Notebook:** `notebooks/module-1/1.5_personal_chef.ipynb`

## Purpose
Build an ingredient-aware recipe assistant using web search, an agent prompt and conversation checkpointing.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- A model answers; an agent can decide to use tools as part of a loop.
- Prompt structure and typed schemas make behavior more predictable, but do not guarantee correctness.
- Short-term memory requires both a checkpointer and a stable conversation/thread ID.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

### Implementation step 2

Imports the libraries and APIs needed by the following example: from langchain.tools import tool; from typing import Dict, Any; from tavily import TavilyClient. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import tool; from typing import Dict, Any; from tavily import TavilyClient. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

### Implementation step 3

Runs this implementation step. Key statement: `system_prompt = """`.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Runs this implementation step. Key statement: `system_prompt = """`.

system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

### Implementation step 5

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

### Implementation step 6

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)